In [ ]:
# CELL 1 - SETUP: get the code, install it, show the GPU
import importlib, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<YOUR_USER>/<YOUR_REPO>.git"   # TODO student: your GitHub repository
BRANCH = "<BRANCH_OR_TAG>"                                     # TODO student: branch / tag / commit to run
CODE_DATASET = "/kaggle/input/<code-dataset>"                           # fallback: code zip uploaded as a private dataset

if not os.path.isdir("repo"):
    if subprocess.call(["git", "clone", "--branch", BRANCH, REPO_URL, "repo"]) != 0:
        print("git clone failed (no internet?) -> copying the code from the Kaggle dataset instead")
        shutil.copytree(CODE_DATASET, "repo")
os.chdir("repo")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "-r", "requirements.txt"])
importlib.invalidate_caches()
subprocess.call(["nvidia-smi"])


In [ ]:
# CELL 2 - PARAMETERS: the only cell you normally edit
TASK = "t1"                                  # t1 | t2cls | t2spec | t3 | t4
MODE = "train"                               # "train" (run_training), "tune" (run_study) or "benchmark" (time the GPU, first thing to run)
CONFIG = "configs/task1_universal.yaml"      # task config file
DEVICE_PROFILE = "kaggle"                 # picks configs/devices/kaggle.yaml
RESUME = "auto"                              # "auto" = newest ckpt_last.pt in the persistent folder; None = fresh run; or a path
OVERRIDES = {}                              # optional config overrides, e.g. {"train.max_steps": 50, "run.smoke": True}


In [ ]:
# CELL 3 - RUN: call the repository code (no logic lives in this notebook)
import subprocess, sys
from genai.tasks.cli import build_config, entry_point

cfg = build_config(TASK, CONFIG, DEVICE_PROFILE, [f"{k}={v}" for k, v in OVERRIDES.items()])

def on_checkpoint(path):
    pass          # Kaggle: /kaggle/working is already persisted by "Save Version"

if MODE == "benchmark":
    # times a few batch sizes with/without AMP and appends rows to docs/BENCHMARKS.md (copy those rows back to your PC)
    subprocess.check_call([sys.executable, "scripts/benchmark.py", "--model", "t1", "--batch", "64", "128", "256", "512",
                           "--device-profile", DEVICE_PROFILE])
    result = "docs/BENCHMARKS.md"
elif MODE == "train":
    result = entry_point(TASK, "train", "run_training")(cfg, resume=RESUME, on_checkpoint=on_checkpoint)
else:
    result = entry_point(TASK, "tune", "run_study")(cfg, on_checkpoint=on_checkpoint)
print("finished:", result)


In [ ]:
# CELL 4 - OUTPUTS: what to download (right-click files in the Kaggle "Output" tab, or Save Version)
from pathlib import Path
out = Path(cfg["output_root"])
for pattern in ("runs/*/*/ckpt_best.pt", "runs/*/*/ckpt_last.pt", "runs/*/*/metrics.jsonl", "runs/*/*/samples/*.png", "optuna/*.db"):
    for p in sorted(out.glob(pattern)):
        print(f"{p.stat().st_size / 2**20:8.1f} MB  {p}")
print("Put the run folder under artifacts/runs/ and the .db under artifacts/optuna/ on your PC.")
